# Wildfire ConvLSTM — Google Colab GPU Training

**Before running:** `Runtime → Change runtime type → T4 GPU`

## GPU Memory Guide

| GPU | VRAM | Safe batch size | Est. epoch time (stride=10) |
|-----|------|-----------------|------------------------------|
| T4  | 16 GB | 8 | ~8 min |
| L4  | 24 GB | 16 | ~3 min |
| A100 | 40 GB | 16 | ~1.5 min |

Activation memory scales linearly with batch size (~3 GB per 4 samples at T=20, H=W=100).

## Workflow
1. Run **Cell 1–6** once per session (GPU check → dataset copy)
2. Run **Cell 7** to start fresh training
3. After any disconnect: re-run **Cells 1–6**, then run **Cell 8** to resume

In [ ]:
# ── Cell 1: GPU Check ────────────────────────────────────────────────────────
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to Runtime → Change runtime type → Hardware accelerator → GPU"
    )

props   = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
print(f"GPU     : {props.name}")
print(f"VRAM    : {vram_gb:.1f} GB")

if   vram_gb >= 35: _auto_bs = 16
elif vram_gb >= 20: _auto_bs = 16
elif vram_gb >= 14: _auto_bs =  8
else:               _auto_bs =  4
print(f"Recommended batch size: {_auto_bs}")

In [ ]:
# ── Cell 2: Clone Repo + Install Dependencies ────────────────────────────────
import os

REPO_URL = "https://github.com/malihashar/wildfire-drone.git"
REPO_DIR = "/content/wildfire-drone"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

# torch, numpy, tqdm already on Colab; install any extras
!pip install tqdm -q

print(f"\nWorking directory: {os.getcwd()}")

In [ ]:
# ── Cell 3: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted at /content/drive")

In [ ]:
# ── Cell 4: Configure Paths & Hyperparameters ────────────────────────────────
#
# EDIT THIS CELL to match your Google Drive folder structure.
#
# Expected Drive layout:
#   MyDrive/
#   └── wildfire-drone-data/
#       └── dataset/
#           ├── normalization.json
#           ├── metadata/
#           │   ├── train.json
#           │   ├── val.json
#           │   └── test.json
#           └── simulations/
#               ├── sim_00002.pt
#               └── ...

DRIVE_ROOT        = "/content/drive/MyDrive"
DRIVE_DATASET     = f"{DRIVE_ROOT}/wildfire-drone-data/dataset"
DRIVE_CHECKPOINTS = f"{DRIVE_ROOT}/wildfire-checkpoints"   # checkpoints saved here

LOCAL_DATASET     = "/content/dataset"   # copied here at session start

# ── Training hyperparameters ─────────────────────────────────────────────────
EPOCHS      = 30
BATCH_SIZE  = 8    # T4=8, L4/A100=16; reduce to 4 if you get OOM
STRIDE      = 10   # window stride; 10 gives ~13k windows (vs 130k at stride=1)
NUM_WORKERS = 2

# Set to an integer (e.g. 100) to copy only that many simulation files.
# Useful for Colab Free where sessions are short (~90 min).
# Set to None to copy all simulations (recommended for Colab Pro).
MAX_SIMS = None

print("Configuration:")
print(f"  Drive dataset : {DRIVE_DATASET}")
print(f"  Drive ckpts   : {DRIVE_CHECKPOINTS}")
print(f"  Local dataset : {LOCAL_DATASET}")
print(f"  Epochs        : {EPOCHS}")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Stride        : {STRIDE}")
print(f"  Num workers   : {NUM_WORKERS}")
print(f"  Max sims      : {MAX_SIMS if MAX_SIMS else 'all'}")

In [ ]:
# ── Cell 5: Copy Dataset from Drive to Local SSD ─────────────────────────────
#
# Colab local disk (NVMe) is ~100x faster than Drive mount for random reads.
# This copy is a one-time cost per session (~20-40 min for 73 GB).
# Already-copied files are skipped so re-running is safe.

import shutil
import time
from pathlib import Path
from tqdm import tqdm

src = Path(DRIVE_DATASET)
dst = Path(LOCAL_DATASET)

if not src.exists():
    raise FileNotFoundError(
        f"Drive dataset not found: {src}\n"
        "Check DRIVE_DATASET in Cell 4 and verify your Drive folder structure."
    )

# Copy normalization + metadata (tiny files)
dst.mkdir(parents=True, exist_ok=True)
(dst / "metadata").mkdir(exist_ok=True)

shutil.copy2(src / "normalization.json", dst / "normalization.json")
print("  ✓ normalization.json")

for split in ["train", "val", "test"]:
    shutil.copy2(src / "metadata" / f"{split}.json",
                 dst / "metadata"  / f"{split}.json")
    print(f"  ✓ metadata/{split}.json")

# Copy simulation .pt files
sim_src = src / "simulations"
sim_dst = dst / "simulations"
sim_dst.mkdir(exist_ok=True)

all_sims = sorted(sim_src.glob("*.pt"))
if MAX_SIMS is not None:
    all_sims = all_sims[:MAX_SIMS]

total_gb = sum(f.stat().st_size for f in all_sims) / 1e9
print(f"\nCopying {len(all_sims)} simulation files ({total_gb:.1f} GB)")
print("Skipping files already present on local disk.")
if total_gb > 10:
    print(f"Estimated time: {total_gb / 50 * 60:.0f}–{total_gb / 30 * 60:.0f} minutes")

t0 = time.time()
skipped = 0
for f in tqdm(all_sims, unit="file", desc="Simulations"):
    dst_f = sim_dst / f.name
    if dst_f.exists():
        skipped += 1
        continue
    shutil.copy2(f, dst_f)

elapsed = time.time() - t0
copied  = len(list(sim_dst.glob("*.pt")))
print(f"\n✓ {copied} files in {LOCAL_DATASET}/simulations "
      f"({skipped} skipped, {elapsed/60:.1f} min)")

In [ ]:
# ── Cell 6: Verify Dataset ───────────────────────────────────────────────────
import json
from pathlib import Path

root = Path(LOCAL_DATASET)
ok   = True

print("Dataset verification:")
for split in ["train", "val", "test"]:
    p = root / "metadata" / f"{split}.json"
    if not p.exists():
        print(f"  ✗ Missing: {p}"); ok = False; continue
    n = len(json.load(open(p)))
    print(f"  ✓ {split}: {n} simulations")

sims = list((root / "simulations").glob("*.pt"))
print(f"  ✓ simulation files : {len(sims)}")

norm = root / "normalization.json"
print(f"  {'✓' if norm.exists() else '✗'} normalization.json")

if not ok:
    raise RuntimeError("Dataset incomplete — fix errors above before training.")

print("\n✓ Dataset ready for training")

In [ ]:
# ── Cell 7: Start Fresh Training ─────────────────────────────────────────────
#
# Run this cell for a NEW training run.
# Checkpoints are saved directly to Google Drive so they survive disconnects.
# Progress is printed every 20 batches.

import os
os.chdir("/content/wildfire-drone")
os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)

cmd = (
    f"python src/train.py"
    f" --focal"
    f" --epochs {EPOCHS}"
    f" --batch_size {BATCH_SIZE}"
    f" --stride {STRIDE}"
    f" --num_workers {NUM_WORKERS}"
    f" --dataset_root {LOCAL_DATASET}"
    f" --checkpoint {DRIVE_CHECKPOINTS}"
)
print("Command:", cmd)
print()
!{cmd}

In [ ]:
# ── Cell 8: Resume Training After Disconnect ──────────────────────────────────
#
# After a Colab session reset:
#   1. Re-run Cells 1–6 (GPU check, clone, mount Drive, config, copy dataset)
#   2. Run THIS cell — it picks up from the last completed epoch.
#
# latest_model.pt on Drive contains epoch number, model weights,
# optimizer state, scheduler state, and full training history.

import os
os.chdir("/content/wildfire-drone")

RESUME_CKPT = f"{DRIVE_CHECKPOINTS}/latest_model.pt"

if not os.path.exists(RESUME_CKPT):
    raise FileNotFoundError(
        f"No checkpoint found at {RESUME_CKPT}\n"
        "Run Cell 7 to start a fresh training run."
    )

cmd = (
    f"python src/train.py"
    f" --focal"
    f" --epochs {EPOCHS}"
    f" --batch_size {BATCH_SIZE}"
    f" --stride {STRIDE}"
    f" --num_workers {NUM_WORKERS}"
    f" --dataset_root {LOCAL_DATASET}"
    f" --checkpoint {DRIVE_CHECKPOINTS}"
    f" --resume {RESUME_CKPT}"
)
print("Resuming from:", RESUME_CKPT)
print("Command:", cmd)
print()
!{cmd}